In [1]:
import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks

import lightgbm as lgb

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

In [2]:
TRAIN_DIR = "/kaggle/input/datasets/cheukhongtang/ass3data/train/train"
TEST_DIR = "/kaggle/input/datasets/cheukhongtang/ass3data/test/test"
SUB_PATH = "/kaggle/input/datasets/cheukhongtang/ass3data/sample_submission.csv"

train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, "User_*", "*.csv")))
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "User_*", "*.csv")))

print("Train files:", len(train_files))
print("Test files:", len(test_files))
print("Sample submission exists:", os.path.exists(SUB_PATH))

Train files: 11020
Test files: 6849
Sample submission exists: True


In [3]:
def get_user_from_path(path):
    parts = path.replace("\\", "/").split("/")
    for p in parts:
        if p.startswith("User_"):
            return p
    return "Unknown"


def safe_skew(x):
    if np.std(x) < 1e-12:
        return 0.0
    return skew(x, nan_policy="omit")


def safe_kurtosis(x):
    if np.std(x) < 1e-12:
        return 0.0
    return kurtosis(x, nan_policy="omit")


def add_basic_signal_features(df):
    df = df.copy()

    df["acc_mag"] = np.sqrt(df["mean_x"]**2 + df["mean_y"]**2 + df["mean_z"]**2)
    df["std_mag"] = np.sqrt(df["std_x"]**2 + df["std_y"]**2 + df["std_z"]**2)

    df["xy_mag"] = np.sqrt(df["mean_x"]**2 + df["mean_y"]**2)
    df["xz_mag"] = np.sqrt(df["mean_x"]**2 + df["mean_z"]**2)
    df["yz_mag"] = np.sqrt(df["mean_y"]**2 + df["mean_z"]**2)

    df["std_xy_mag"] = np.sqrt(df["std_x"]**2 + df["std_y"]**2)
    df["std_xz_mag"] = np.sqrt(df["std_x"]**2 + df["std_z"]**2)
    df["std_yz_mag"] = np.sqrt(df["std_y"]**2 + df["std_z"]**2)

    df["mean_sum"] = df["mean_x"] + df["mean_y"] + df["mean_z"]
    df["std_sum"] = df["std_x"] + df["std_y"] + df["std_z"]

    return df

In [4]:
def add_stat_features(row, values, prefix):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        values = np.array([0.0])

    q05, q10, q25, q75, q90, q95 = np.quantile(
        values, [0.05, 0.10, 0.25, 0.75, 0.90, 0.95]
    )

    mean_val = np.mean(values)

    stats = {
        "mean": mean_val,
        "std": np.std(values),
        "min": np.min(values),
        "max": np.max(values),
        "range": np.max(values) - np.min(values),
        "median": np.median(values),
        "q05": q05,
        "q10": q10,
        "q25": q25,
        "q75": q75,
        "q90": q90,
        "q95": q95,
        "iqr": q75 - q25,
        "energy": np.mean(values ** 2),
        "rms": np.sqrt(np.mean(values ** 2)),
        "mad": np.mean(np.abs(values - mean_val)),
        "skew": safe_skew(values),
        "kurtosis": safe_kurtosis(values),
    }

    for name, value in stats.items():
        row[f"{prefix}_{name}"] = value


def add_diff_features(row, values, prefix):
    values = np.asarray(values, dtype=float)

    if len(values) < 2:
        diffs = np.array([0.0])
    else:
        diffs = np.diff(values)

    row[f"{prefix}_diff_mean"] = np.mean(diffs)
    row[f"{prefix}_diff_std"] = np.std(diffs)
    row[f"{prefix}_diff_abs_mean"] = np.mean(np.abs(diffs))
    row[f"{prefix}_diff_abs_max"] = np.max(np.abs(diffs))
    row[f"{prefix}_diff_energy"] = np.mean(diffs ** 2)

    row[f"{prefix}_first_60_mean"] = np.mean(values[:60])
    row[f"{prefix}_last_60_mean"] = np.mean(values[-60:])
    row[f"{prefix}_last_minus_first_60"] = np.mean(values[-60:]) - np.mean(values[:60])

In [5]:
def add_fft_features(row, values, prefix, sample_rate=1.0):
    values = np.asarray(values, dtype=float)
    values = values - np.mean(values)

    if len(values) < 4 or np.std(values) < 1e-12:
        row[f"{prefix}_fft_dom_freq"] = 0.0
        row[f"{prefix}_fft_dom_power"] = 0.0
        row[f"{prefix}_fft_total_power"] = 0.0
        row[f"{prefix}_fft_spectral_entropy"] = 0.0
        row[f"{prefix}_fft_low_power"] = 0.0
        row[f"{prefix}_fft_high_power"] = 0.0
        return

    fft_vals = np.fft.rfft(values)
    power = np.abs(fft_vals) ** 2
    freqs = np.fft.rfftfreq(len(values), d=1.0 / sample_rate)

    power_no_dc = power[1:]
    freqs_no_dc = freqs[1:]

    if len(power_no_dc) == 0 or power_no_dc.sum() <= 0:
        row[f"{prefix}_fft_dom_freq"] = 0.0
        row[f"{prefix}_fft_dom_power"] = 0.0
        row[f"{prefix}_fft_total_power"] = 0.0
        row[f"{prefix}_fft_spectral_entropy"] = 0.0
        row[f"{prefix}_fft_low_power"] = 0.0
        row[f"{prefix}_fft_high_power"] = 0.0
        return

    dom_idx = np.argmax(power_no_dc)
    total_power = np.sum(power_no_dc)
    prob = power_no_dc / total_power

    low_mask = freqs_no_dc <= 0.10
    high_mask = freqs_no_dc > 0.10

    row[f"{prefix}_fft_dom_freq"] = freqs_no_dc[dom_idx]
    row[f"{prefix}_fft_dom_power"] = power_no_dc[dom_idx]
    row[f"{prefix}_fft_total_power"] = total_power
    row[f"{prefix}_fft_spectral_entropy"] = -np.sum(prob * np.log(prob + 1e-12))
    row[f"{prefix}_fft_low_power"] = np.sum(power_no_dc[low_mask])
    row[f"{prefix}_fft_high_power"] = np.sum(power_no_dc[high_mask])


def add_peak_features(row, values, prefix):
    values = np.asarray(values, dtype=float)

    if len(values) < 3 or np.std(values) < 1e-12:
        row[f"{prefix}_num_peaks"] = 0
        row[f"{prefix}_peak_rate"] = 0.0
        row[f"{prefix}_mean_peak_height"] = 0.0
        row[f"{prefix}_max_peak_height"] = 0.0
        return

    height_threshold = np.mean(values) + 0.5 * np.std(values)
    peaks, properties = find_peaks(values, height=height_threshold)

    row[f"{prefix}_num_peaks"] = len(peaks)
    row[f"{prefix}_peak_rate"] = len(peaks) / len(values)

    if len(peaks) > 0:
        heights = properties["peak_heights"]
        row[f"{prefix}_mean_peak_height"] = np.mean(heights)
        row[f"{prefix}_max_peak_height"] = np.max(heights)
    else:
        row[f"{prefix}_mean_peak_height"] = 0.0
        row[f"{prefix}_max_peak_height"] = 0.0

In [6]:
def add_window_features(row, df, cols, n_windows=10):
    n = len(df)
    window_size = n // n_windows

    for w in range(n_windows):
        start = w * window_size
        end = (w + 1) * window_size if w < n_windows - 1 else n
        window = df.iloc[start:end]

        for col in cols:
            values = window[col].values
            row[f"win{w}_{col}_mean"] = np.mean(values)
            row[f"win{w}_{col}_std"] = np.std(values)
            row[f"win{w}_{col}_min"] = np.min(values)
            row[f"win{w}_{col}_max"] = np.max(values)
            row[f"win{w}_{col}_energy"] = np.mean(values ** 2)

In [7]:
def summarize_file(path, is_train=True):
    df = pd.read_csv(path)
    df = df.sort_values("index").reset_index(drop=True)
    df = df.interpolate().ffill().bfill()
    df = add_basic_signal_features(df)

    row = {}
    row["path"] = path
    row["user"] = get_user_from_path(path)
    row["filename"] = os.path.basename(path)

    if "file_id" in df.columns:
        row["file_id"] = int(df["file_id"].iloc[0])
    else:
        row["file_id"] = int(os.path.splitext(os.path.basename(path))[0])

    if is_train:
        row["label"] = int(df["label"].iloc[0])

    base_cols = [
        "mean_x", "mean_y", "mean_z",
        "std_x", "std_y", "std_z",
        "acc_mag", "std_mag",
        "xy_mag", "xz_mag", "yz_mag",
        "std_xy_mag", "std_xz_mag", "std_yz_mag",
        "mean_sum", "std_sum"
    ]

    for col in base_cols:
        values = df[col].values
        add_stat_features(row, values, col)
        add_diff_features(row, values, col)

    for col in ["acc_mag", "std_mag", "std_x", "std_y", "std_z"]:
        values = df[col].values
        add_fft_features(row, values, col)
        add_peak_features(row, values, col)

    add_window_features(
        row,
        df,
        cols=["acc_mag", "std_mag", "std_x", "std_y", "std_z"],
        n_windows=10
    )

    return row


def build_feature_table(files, is_train=True):
    rows = []
    for path in tqdm(files):
        rows.append(summarize_file(path, is_train=is_train))
    return pd.DataFrame(rows)

In [8]:
train_features = build_feature_table(train_files, is_train=True)
test_features = build_feature_table(test_files, is_train=False)

print("Train features:", train_features.shape)
print("Test features:", test_features.shape)

train_features.to_csv("train_full_features_for_ablation.csv", index=False)
test_features.to_csv("test_full_features_for_ablation.csv", index=False)

  0%|          | 0/11020 [00:00<?, ?it/s]

  0%|          | 0/6849 [00:00<?, ?it/s]

Train features: (11020, 721)
Test features: (6849, 720)


In [9]:
id_cols = ["path", "user", "filename", "file_id"]
target_col = "label"

feature_cols = [
    c for c in train_features.columns
    if c not in id_cols + [target_col]
]

X = train_features[feature_cols].copy()
y = train_features[target_col].copy()
groups = train_features["user"].copy()

X = X.replace([np.inf, -np.inf], np.nan)
medians = X.median()
X = X.fillna(medians)

sample_submission = pd.read_csv(SUB_PATH)

test_features_ordered = sample_submission[["Id"]].merge(
    test_features,
    left_on="Id",
    right_on="file_id",
    how="left"
)

print("Missing test matches:", test_features_ordered["file_id"].isna().sum())

X_test_ordered = test_features_ordered[feature_cols].copy()
X_test_ordered = X_test_ordered.replace([np.inf, -np.inf], np.nan)
X_test_ordered = X_test_ordered.fillna(medians)

print("X:", X.shape)
print("X_test_ordered:", X_test_ordered.shape)

Missing test matches: 0
X: (11020, 716)
X_test_ordered: (6849, 716)


In [10]:
def select_features(feature_cols, remove_patterns=None, keep_patterns=None):
    selected = []

    for col in feature_cols:
        keep = True

        if keep_patterns is not None:
            keep = any(p in col for p in keep_patterns)

        if remove_patterns is not None:
            if any(p in col for p in remove_patterns):
                keep = False

        if keep:
            selected.append(col)

    return selected


feature_sets = {}

feature_sets["full_lightgbm"] = list(feature_cols)

feature_sets["basic_stats_only"] = [
    c for c in feature_cols
    if any(c.endswith(suffix) for suffix in ["_mean", "_std", "_min", "_max", "_range", "_median"])
    and not c.startswith("win")
    and "_diff_" not in c
    and "_fft_" not in c
    and "_peak" not in c
    and "first_60" not in c
    and "last_60" not in c
    and "last_minus_first" not in c
]

feature_sets["no_windows"] = [
    c for c in feature_cols
    if not c.startswith("win")
]

feature_sets["no_fft"] = select_features(
    feature_cols,
    remove_patterns=["_fft_"]
)

feature_sets["no_peaks"] = select_features(
    feature_cols,
    remove_patterns=[
        "_num_peaks",
        "_peak_rate",
        "_mean_peak_height",
        "_max_peak_height"
    ]
)

feature_sets["no_temporal_diff_trend"] = select_features(
    feature_cols,
    remove_patterns=[
        "_diff_",
        "first_60",
        "last_60",
        "last_minus_first"
    ]
)

feature_sets["no_magnitude"] = select_features(
    feature_cols,
    remove_patterns=[
        "acc_mag",
        "std_mag",
        "xy_mag",
        "xz_mag",
        "yz_mag",
        "std_xy_mag",
        "std_xz_mag",
        "std_yz_mag",
        "mean_sum",
        "std_sum"
    ]
)

for name, cols in feature_sets.items():
    print(name, len(cols))

full_lightgbm 716
basic_stats_only 96
no_windows 466
no_fft 686
no_peaks 696
no_temporal_diff_trend 588
no_magnitude 336


In [11]:
def run_lightgbm_ablation(
    experiment_name,
    selected_features,
    X,
    y,
    groups,
    X_test_ordered,
    sample_submission,
    n_splits=5,
    random_seed=42
):
    print("=" * 80)
    print("Experiment:", experiment_name)
    print("Number of features:", len(selected_features))
    print("=" * 80)

    X_exp = X[selected_features].copy()
    X_test_exp = X_test_ordered[selected_features].copy()

    X_exp = X_exp.replace([np.inf, -np.inf], np.nan)
    X_test_exp = X_test_exp.replace([np.inf, -np.inf], np.nan)

    exp_medians = X_exp.median()
    X_exp = X_exp.fillna(exp_medians)
    X_test_exp = X_test_exp.fillna(exp_medians)

    gkf = GroupKFold(n_splits=n_splits)

    oof_preds = np.zeros(len(X_exp), dtype=int)
    test_proba = np.zeros((len(X_test_exp), 6))

    fold_scores = []
    models = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_exp, y, groups), start=1):
        print(f"\n{experiment_name} - Fold {fold}")

        X_train = X_exp.iloc[train_idx]
        X_val = X_exp.iloc[val_idx]
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        sample_weight = compute_sample_weight(
            class_weight="balanced",
            y=y_train
        )

        model = lgb.LGBMClassifier(
            objective="multiclass",
            num_class=6,
            n_estimators=2000,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=20,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=random_seed + fold,
            n_jobs=-1,
            verbose=-1
        )

        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weight,
            eval_set=[(X_val, y_val)],
            eval_metric="multi_logloss",
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=100)
            ]
        )

        val_proba = model.predict_proba(X_val)
        val_pred = np.argmax(val_proba, axis=1)

        oof_preds[val_idx] = val_pred

        fold_score = f1_score(y_val, val_pred, average="macro")
        fold_scores.append(fold_score)

        print(f"Fold {fold} macro-F1: {fold_score:.5f}")

        test_proba += model.predict_proba(X_test_exp) / n_splits
        models.append(model)

    mean_macro_f1 = np.mean(fold_scores)
    std_macro_f1 = np.std(fold_scores)
    oof_macro_f1 = f1_score(y, oof_preds, average="macro")

    print("\nExperiment:", experiment_name)
    print("Mean macro-F1:", mean_macro_f1)
    print("Std macro-F1:", std_macro_f1)
    print("OOF macro-F1:", oof_macro_f1)
    print(classification_report(y, oof_preds, digits=4))

    test_pred = np.argmax(test_proba, axis=1)

    submission = sample_submission.copy()
    submission["Label"] = test_pred.astype(int)

    output_name = f"submission_ablation_{experiment_name}.csv"
    submission.to_csv(output_name, index=False)

    print("Saved:", output_name)
    print("Prediction distribution:")
    display(submission["Label"].value_counts().sort_index())

    return {
        "experiment": experiment_name,
        "n_features": len(selected_features),
        "fold_scores": fold_scores,
        "mean_macro_f1": mean_macro_f1,
        "std_macro_f1": std_macro_f1,
        "oof_macro_f1": oof_macro_f1,
        "submission_file": output_name
    }

In [12]:
selected_experiments = [
    "full_lightgbm",
    "basic_stats_only",
    "no_windows",
    "no_fft",
    "no_peaks",
    "no_temporal_diff_trend",
    "no_magnitude"
]

ablation_results = []

for experiment_name in selected_experiments:
    result = run_lightgbm_ablation(
        experiment_name=experiment_name,
        selected_features=feature_sets[experiment_name],
        X=X,
        y=y,
        groups=groups,
        X_test_ordered=X_test_ordered,
        sample_submission=sample_submission,
        n_splits=5,
        random_seed=42
    )

    ablation_results.append(result)

Experiment: full_lightgbm
Number of features: 716

full_lightgbm - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.404561
[200]	valid_0's multi_logloss: 0.378068
Early stopping, best iteration is:
[177]	valid_0's multi_logloss: 0.377484
Fold 1 macro-F1: 0.67125

full_lightgbm - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.363374
[200]	valid_0's multi_logloss: 0.338949
Early stopping, best iteration is:
[170]	valid_0's multi_logloss: 0.336633
Fold 2 macro-F1: 0.75795

full_lightgbm - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.326016
[200]	valid_0's multi_logloss: 0.31
Early stopping, best iteration is:
[155]	valid_0's multi_logloss: 0.304925
Fold 3 macro-F1: 0.74518

full_lightgbm - Fold 4
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.407915
[200]	valid_0's multi_logloss:

Label
0    2813
1    3074
2      98
3     505
4      95
5     264
Name: count, dtype: int64

Experiment: basic_stats_only
Number of features: 96

basic_stats_only - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.499013
[200]	valid_0's multi_logloss: 0.457196
[300]	valid_0's multi_logloss: 0.463572
Early stopping, best iteration is:
[218]	valid_0's multi_logloss: 0.455857
Fold 1 macro-F1: 0.64660

basic_stats_only - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.429631
[200]	valid_0's multi_logloss: 0.398506
[300]	valid_0's multi_logloss: 0.406374
Early stopping, best iteration is:
[224]	valid_0's multi_logloss: 0.397814
Fold 2 macro-F1: 0.71249

basic_stats_only - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.383975
[200]	valid_0's multi_logloss: 0.348215
Early stopping, best iteration is:
[198]	valid_0's multi_logloss: 0.348062
Fold 3 macro-F1: 0.73100

basic_stats_only - Fold 4
Training until validation scores d

Label
0    2857
1    2996
2     120
3     531
4      83
5     262
Name: count, dtype: int64

Experiment: no_windows
Number of features: 466

no_windows - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.415325
[200]	valid_0's multi_logloss: 0.387505
Early stopping, best iteration is:
[180]	valid_0's multi_logloss: 0.38721
Fold 1 macro-F1: 0.67856

no_windows - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.374221
[200]	valid_0's multi_logloss: 0.349463
Early stopping, best iteration is:
[165]	valid_0's multi_logloss: 0.346585
Fold 2 macro-F1: 0.75246

no_windows - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.341735
[200]	valid_0's multi_logloss: 0.330887
Early stopping, best iteration is:
[147]	valid_0's multi_logloss: 0.325667
Fold 3 macro-F1: 0.72665

no_windows - Fold 4
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.415529
[200]	valid_0's multi_logloss: 0.418558
Ea

...

Label
0    2815
1    3074
2      93
3     518
4      91
5     258
Name: count, dtype: int64

Experiment: no_temporal_diff_trend
Number of features: 588

no_temporal_diff_trend - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.408987
[200]	valid_0's multi_logloss: 0.382594
Early stopping, best iteration is:
[181]	valid_0's multi_logloss: 0.381988
Fold 1 macro-F1: 0.66728

no_temporal_diff_trend - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.368403
[200]	valid_0's multi_logloss: 0.342396
Early stopping, best iteration is:
[168]	valid_0's multi_logloss: 0.340131
Fold 2 macro-F1: 0.75683

no_temporal_diff_trend - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.332353
[200]	valid_0's multi_logloss: 0.317382
Early stopping, best iteration is:
[153]	valid_0's multi_logloss: 0.313929
Fold 3 macro-F1: 0.73545

no_temporal_diff_trend - Fold 4
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi

Label
0    2816
1    3078
2      92
3     509
4      96
5     258
Name: count, dtype: int64

Experiment: no_magnitude
Number of features: 336

no_magnitude - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.416779
[200]	valid_0's multi_logloss: 0.391501
Early stopping, best iteration is:
[194]	valid_0's multi_logloss: 0.390575
Fold 1 macro-F1: 0.65557

no_magnitude - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.386081
[200]	valid_0's multi_logloss: 0.343877
[300]	valid_0's multi_logloss: 0.350141
Early stopping, best iteration is:
[216]	valid_0's multi_logloss: 0.342851
Fold 2 macro-F1: 0.73236

no_magnitude - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.322225
[200]	valid_0's multi_logloss: 0.291755
Early stopping, best iteration is:
[186]	valid_0's multi_logloss: 0.291631
Fold 3 macro-F1: 0.72918

no_magnitude - Fold 4
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 

Label
0    2810
1    3077
2     100
3     542
4      70
5     250
Name: count, dtype: int64

In [13]:
ablation_summary = pd.DataFrame([
    {
        "experiment": r["experiment"],
        "n_features": r["n_features"],
        "mean_macro_f1": r["mean_macro_f1"],
        "std_macro_f1": r["std_macro_f1"],
        "oof_macro_f1": r["oof_macro_f1"],
        "submission_file": r["submission_file"]
    }
    for r in ablation_results
])

full_score = ablation_summary.loc[
    ablation_summary["experiment"] == "full_lightgbm",
    "mean_macro_f1"
].iloc[0]

ablation_summary["drop_vs_full"] = full_score - ablation_summary["mean_macro_f1"]

ablation_summary = ablation_summary.sort_values(
    "mean_macro_f1",
    ascending=False
).reset_index(drop=True)

display(ablation_summary)

ablation_summary.to_csv("ablation_summary.csv", index=False)
print("Saved ablation_summary.csv")

,experiment,n_features,mean_macro_f1,std_macro_f1,oof_macro_f1,submission_file,drop_vs_full
0,full_lightgbm,716,0.720552,0.030085,0.729860,submission_ablation_full_lightgbm.csv,0.000000
1,no_fft,686,0.717770,0.030407,0.727073,submission_ablation_no_fft.csv,0.002782
2,no_peaks,696,0.717715,0.030146,0.727297,submission_ablation_no_peaks.csv,0.002837
3,no_windows,466,0.715101,0.025082,0.725414,submission_ablation_no_windows.csv,0.005452
4,no_temporal_diff_trend,588,0.713840,0.030628,0.723353,submission_ablation_no_temporal_diff_trend.csv,0.006713
5,no_magnitude,336,0.706138,0.030264,0.717215,submission_ablation_no_magnitude.csv,0.014415
6,basic_stats_only,96,0.691403,0.029331,0.704503,submission_ablation_basic_stats_only.csv,0.029150


Saved ablation_summary.csv
